In [0]:

default_key_cols="execution_id,model,layer,topic"
instruction_activity_list=["model_evaluation", "topic_naming"]
refinement_queue_table_name="topics_to_refine"
refinement_min_topic_count=100
refinement_score_threshold=4
refinement_max_layer=1
checkpoint_path_prefix_template = "/Volumes/{catalog}/{schema}/checkpoints/{stream_table_name}"
cluster_model_list=["HDBSCAN", "KMEANS"]
write_mode_list=["append", "overwrite"]
execution_keys_list=["cluster_model", "timestamp"]
catalog_list=["amit"]
embedding_model_list=["databricks-qwen3-embedding-0-6b"]
schema_list=["bertopic","default"]
input_table_list=["bertopic_input"]
model_result_table_list=["topic_info_local"]
instruct_model_list=["databricks-meta-llama-3-3-70b-instruct", "databricks-qwen3-next-80b-a3b-instruct"]
reduced_dimensions_list=["5","10","15"]
from pyspark.sql.functions import col
def add_column(table_name, column_name, column_type):
    try:
        spark.sql(f"DESCRIBE {table_name}").filter(col("col_name") == column_name).count()
        print(f"Column {column_name} already exists")
    except:
        spark.sql(f"""
        ALTER TABLE {table_name}
        ADD COLUMNS ({column_name} {column_type})
        """)
        print(f"Column {column_name} added")

def get_ai_merge_sql(table_name, model_name, ai_result_column, input_column ):
  sql_str=f"""  MERGE INTO {table_name} t
    USING (
      SELECT
        id,
        ai_query("{model_name}", {input_column}) AS embedding
      FROM {table_name}
      WHERE {ai_result_column} IS NULL
        AND {input_column} IS NOT NULL and {input_column}!= ''
      LIMIT 300
    ) s
    ON t.id = s.id
    WHEN MATCHED AND t.{ai_result_column} IS NULL THEN
      UPDATE SET t.{ai_result_column} = s.embedding;
  """
  return sql_str
import time
def add_column_batch(table_name, model_name,ai_result_column,input_column, column_type, is_ai ):
    add_column(table_name, column_name=ai_result_column, column_type=column_type)
    sql_str=None
    if is_ai==True:
        sql_str=get_ai_merge_sql(table_name=table_name, model_name=model_name, ai_result_column=ai_result_column, input_column=input_column)
    batch_num = 0
    while True:
        null_count = spark.sql(f"""
            SELECT COUNT(*) AS c
            FROM {table_name}
            WHERE {ai_result_column} IS NULL
            AND {input_column} IS NOT NULL and {input_column} <>""
        """).collect()[0]["c"]
        batch_num += 1
        print(f"Remaining records to process: {null_count}. Processing batch {batch_num}...")
        if null_count == 0:
            print(f"Done. No more NULL {ai_result_column}.")
            break

        start = time.perf_counter()
        spark.sql(sql_str)  
        elapsed = time.perf_counter() - start
        print(f"Batch {batch_num} latency: {elapsed:.2f} sec")  

from delta.tables import DeltaTable

def create_table_with_cdf(table_name, df):
    DeltaTable.createIfNotExists(spark) \
        .tableName(table_name) \
        .addColumns(df.schema) \
        .property("delta.enableChangeDataFeed", "true") \
        .execute()
def create_table(table_name, df):
    df.write.mode("overwrite").saveAsTable(table_name)
def get_table(table_name):
    return DeltaTable.forName(spark, table_name)
def get_table_properties(table_name):
    return get_table(table_name).properties 
def write_to_table(table_name, df, write_mode="append", overwriteSchema="true"):
    create_table_with_cdf(table_name, df )
    df.write \
      .mode(write_mode) \
      .option("mergeSchema", "true") \
      .saveAsTable(table_name)

def read_checkpoint(path: str) -> int | None:
    try:
        value = dbutils.fs.head(path).strip()
        return int(value)
    except Exception:
        return None

def write_checkpoint(path: str, version: int) -> None:
    dbutils.fs.put(path, str(version), overwrite=True)
def get_current_version(table_name):
    return (
        spark.sql(f"DESCRIBE HISTORY {table_name}")
        .selectExpr("max(version) as version")
        .first()["version"]
    )
